# Training Dataset EDA
Exploratory analysis of the OptoJump annotation dataset used for model training.

In [ ]:
import os, json
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

ANNOTATIONS_CSV = "./data/output/annotations/optojump/ml_training_dataset.csv"
RESULTS_JSON    = "./data/output/gait/pose_data/stage1/results.json"
PLOTS_DIR       = "./data/output/gait/plots/eda"
RECORDING_FPS   = 120

os.makedirs(PLOTS_DIR, exist_ok=True)

In [ ]:
raw = pd.read_csv(ANNOTATIONS_CSV)

# Derive athlete name, study id and test id from video_path
raw["study"]   = raw["video_path"].str.extract(r"study_(\d+)").astype(int)
raw["athlete"] = raw["video_path"].apply(
    lambda p: "_".join(os.path.splitext(os.path.basename(p))[0].split("_")[:-1])
)
raw["test_id"] = raw["video_path"].apply(
    lambda p: int(os.path.splitext(os.path.basename(p))[0].split("_")[-1])
)

# Label contact bouts (consecutive contact frames per video as one step)
contact = raw[raw["label"] == "contact"].copy()
contact["bout"] = (
    contact.groupby("video_path")["frame_number"]
    .transform(lambda x: (x.diff() != 1).cumsum())
)

# ── Train / test split (mirrors train_test_split in dataset.py: seed=42, 10%) ─
_athletes = sorted(raw["athlete"].unique())
_n_test   = int(len(_athletes) * 0.10)
_rng      = np.random.default_rng(42)
TEST_ATHLETES = sorted(_rng.choice(_athletes, _n_test, replace=False).tolist())
_test_set = set(TEST_ATHLETES)

raw_train = raw[~raw["athlete"].isin(_test_set)].copy()
raw_test  = raw[ raw["athlete"].isin(_test_set)].copy()

contact_train = contact[~contact["video_path"].isin(raw_test["video_path"])].copy()
contact_test  = contact[ contact["video_path"].isin(raw_test["video_path"])].copy()

print(f"Total  : {len(raw):,} frames  |  {raw['video_path'].nunique()} videos  |  {raw['athlete'].nunique()} athletes")
print(f"Train  : {len(raw_train):,} frames  |  {raw_train['video_path'].nunique()} videos  |  {raw_train['athlete'].nunique()} athletes")
print(f"Test   : {len(raw_test):,} frames  |  {raw_test['video_path'].nunique()} videos  |  {raw_test['athlete'].nunique()} athletes  ({', '.join(TEST_ATHLETES)})")

## 1 · Dataset overview

In [ ]:
def _subset_stats(df, cont_df, label):
    n_f       = len(df)
    n_contact = (df["label"] == "contact").sum()
    n_flight  = (df["label"] == "flight").sum()
    n_vid     = df["video_path"].nunique()
    n_ath     = df["athlete"].nunique()
    n_steps   = cont_df.groupby("video_path")["bout"].nunique().sum()
    return {
        "Subset":                 label,
        "Total frames":           f"{n_f:,}",
        "Contact frames":         f"{n_contact:,}  ({n_contact/n_f*100:.1f}%)",
        "Flight frames":          f"{n_flight:,}  ({n_flight/n_f*100:.1f}%)",
        "Videos":                 str(n_vid),
        "Athletes":               str(n_ath),
        "Contact bouts (steps)":  str(n_steps),
    }

summary = pd.DataFrame([
    _subset_stats(raw_train, contact_train, "Train"),
    _subset_stats(raw_test,  contact_test,  "Test"),
    _subset_stats(raw,       contact,       "Total"),
]).set_index("Subset")
summary.style.set_caption("Dataset summary — train / test / total")

## 2 · Flight / contact split

In [ ]:
counts = raw_train["label"].value_counts()
fig, ax = plt.subplots()
ax.pie(counts, labels=counts.index, autopct="%1.1f%%")
plt.savefig(f"{PLOTS_DIR}/label_split.png", bbox_inches="tight")
plt.close()

In [ ]:
video_contact_pct = (
    raw_train.groupby("video_path")["label"]
    .apply(lambda x: (x == "contact").mean() * 100)
    .sort_values()
)
fig, ax = plt.subplots(figsize=(8, 4))
ax.barh(range(len(video_contact_pct)), video_contact_pct.values)
ax.axvline(video_contact_pct.mean(), color="red", ls="--")
ax.set_xlabel("Contact frames (%)")
ax.set_yticks([])
plt.savefig(f"{PLOTS_DIR}/contact_pct_per_video.png", bbox_inches="tight")
plt.close()

## 3 · Athletes

In [ ]:
videos_per_athlete = raw_train.groupby("athlete")["video_path"].nunique().sort_values()
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(videos_per_athlete.index, videos_per_athlete.values)
ax.set_xlabel("Number of videos")
plt.savefig(f"{PLOTS_DIR}/videos_per_athlete.png", bbox_inches="tight")
plt.close()

## 4 · Steps per video

In [ ]:
steps_per_video = contact_train.groupby("video_path")["bout"].nunique()
fig, ax = plt.subplots()
ax.hist(steps_per_video, bins=range(1, steps_per_video.max() + 2), align="left")
ax.axvline(steps_per_video.mean(), color="red", ls="--")
ax.set_xlabel("Steps per video")
ax.set_ylabel("Number of videos")
plt.savefig(f"{PLOTS_DIR}/steps_hist.png", bbox_inches="tight")
plt.close()

In [ ]:
video_athlete = raw_train[["video_path", "athlete"]].drop_duplicates()
steps_athlete = (
    steps_per_video.reset_index()
    .merge(video_athlete, on="video_path")
    .groupby("athlete")["bout"].mean()
    .sort_values()
)
fig, ax = plt.subplots(figsize=(8, 6))
ax.barh(steps_athlete.index, steps_athlete.values)
ax.set_xlabel("Mean steps per video")
plt.savefig(f"{PLOTS_DIR}/steps_per_athlete.png", bbox_inches="tight")
plt.close()

## 5 · Contact duration per step

In [ ]:
bout_lengths = contact_train.groupby(["video_path", "bout"])["frame_number"].count()
bout_ms = bout_lengths / RECORDING_FPS * 1000
fig, ax = plt.subplots()
ax.hist(bout_ms, bins=40)
ax.axvline(bout_ms.mean(), color="red", ls="--")
ax.set_xlabel("Contact duration (ms)")
ax.set_ylabel("Number of steps")
plt.savefig(f"{PLOTS_DIR}/contact_duration.png", bbox_inches="tight")
plt.close()

## 6 · Left / right side balance

In [ ]:
bout_side = (
    contact_train.groupby(["video_path", "bout"])["side"]
    .first().reset_index()
)
side_counts = bout_side["side"].value_counts()
fig, ax = plt.subplots()
ax.pie(side_counts, labels=side_counts.index, autopct="%1.1f%%")
plt.savefig(f"{PLOTS_DIR}/side_balance.png", bbox_inches="tight")
plt.close()

In [ ]:
video_athlete = raw_train[["video_path", "athlete"]].drop_duplicates()
athlete_side = (
    bout_side.merge(video_athlete, on="video_path")
    .groupby(["athlete", "side"]).size().unstack(fill_value=0)
    .reindex(columns=["left", "right"], fill_value=0)
    .sort_values("left")
)
fig, ax = plt.subplots(figsize=(8, 6))
athlete_side.plot(kind="barh", ax=ax, width=0.7)
ax.set_xlabel("Number of contact bouts")
plt.savefig(f"{PLOTS_DIR}/side_per_athlete.png", bbox_inches="tight")
plt.close()

# Feature Selection — XGBoost ablation (train set only)

Results are loaded from `stage1_baselines.json`, which was computed using
leave-one-athlete-out cross-validation on the **train split only**.

In [ ]:
with open(RESULTS_JSON) as f:
    data = json.load(f)

keys   = list(data["xgboost_ablation"])
models = ["kinematic", *[f"ablation_{k}" for k in keys]]

In [ ]:
accuracies = [data["kinematic"]["accuracy"],
              *[data["xgboost_ablation"][k]["accuracy"] for k in keys]]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(models, accuracies)
ax.set_ylabel("Accuracy")
plt.xticks(rotation=45, ha="right")
plt.savefig(f"{PLOTS_DIR}/accuracy_comparison.png", bbox_inches="tight")
plt.close()

In [ ]:
f1_scores = [data["kinematic"]["f1"]["macro"],
             *[data["xgboost_ablation"][k]["f1"]["macro"] for k in keys]]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(models, f1_scores)
ax.set_ylabel("F1-macro")
plt.xticks(rotation=45, ha="right")
plt.savefig(f"{PLOTS_DIR}/f1_comparison.png", bbox_inches="tight")
plt.close()

In [ ]:
labels = ["Left Stance", "Right Stance", "Flight"]
cm = np.array(data["xgboost_ablation"][keys[-1]]["confusion_matrix"], dtype=float)
cm_norm = cm / cm.sum(axis=1, keepdims=True)
fig, ax = plt.subplots(figsize=(5, 4))
sns.heatmap(cm_norm, annot=True, fmt=".2f", xticklabels=labels, yticklabels=labels,
            cbar=False, ax=ax)
ax.set_xlabel("Predicted")
ax.set_ylabel("Actual")
plt.savefig(f"{PLOTS_DIR}/confusion_matrix.png", bbox_inches="tight")
plt.close()

In [ ]:
importances = data["xgboost_ablation"]["D_all"]["feature_importances"]
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(range(len(importances)), importances)
ax.set_xlabel("Feature index")
ax.set_ylabel("Importance")
plt.savefig(f"{PLOTS_DIR}/feature_importances.png", bbox_inches="tight")
plt.close()

### Features:
- 0–5   : norm. y-positions  (L/R heel, big_toe, ankle)
- 6–11  : y-velocities
- 12–15 : x-velocities
- 16–19 : joint angles
- 20–21 : hip y + hip dy/dt

In [ ]:
timing = data["kinematic"]["timing_error"]
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(list(timing), [v["ms"] for v in timing.values()])
ax.set_ylabel("ms")
plt.savefig(f"{PLOTS_DIR}/timing_error.png", bbox_inches="tight")
plt.close()